# Script to compute betas per clusters session averages

## merge function

In [ ]:
import pandas as pd
import re

def load_and_merge_beta_lmer_data(beta_file_path, lmer_file_path, exclude_participants=None, calorie=None):
    """
    Load beta estimates and LMER data, process them, and merge.
    
    Parameters:
    -----------
    beta_file_path : str
        Path to the TSV file containing beta estimates
    lmer_file_path : str
        Path to the TXT file containing LMER data
    exclude_participants : list of str, optional
        List of substrings to exclude from participant_id (e.g., ['RGC'])
    calorie : str, optional
        Tag to add as a 'calorie' column to all rows (e.g., 'high', 'low', None)
    
    Returns:
    --------
    df_merged : pd.DataFrame
        Merged dataframe with beta and LMER data
    df_beta_processed : pd.DataFrame
        Processed beta dataframe with session-run structure
    """
    
    # Load beta TSV file
    print(f"Loading beta data from: {beta_file_path}")
    df_beta = pd.read_csv(beta_file_path, sep='\t')
    print(f"Loaded {len(df_beta)} rows")
    
    # Process session column
    print(f"\nSession dtype: {df_beta['session'].dtype}")
    print(f"Session unique values: {df_beta['session'].unique()}")
    
    # Always remove ses-01 BEFORE converting to int
    if df_beta['session'].dtype == 'object':
        before_count = len(df_beta)
        df_beta = df_beta[df_beta['session'] != 'ses-01']
        after_count = len(df_beta)
        print(f"Removed session 'ses-01': {before_count - after_count} rows removed")
        
        # Now convert to int
        df_beta['session'] = df_beta['session'].str.replace('ses-', '').astype(int)
    else:
        # If already numeric, ensure it's int
        df_beta['session'] = df_beta['session'].astype(int)
    
    # Process run column if it exists and needs cleaning
    if 'run' in df_beta.columns and df_beta['run'].dtype == 'object':
        df_beta['run'] = df_beta['run'].str.replace('run-', '').astype(int)
    elif 'run' in df_beta.columns:
        df_beta['run'] = df_beta['run'].astype(int)
    
    # Exclude participants if specified
    if exclude_participants:
        for exclude_substr in exclude_participants:
            before_count = len(df_beta)
            df_beta = df_beta[~df_beta['participant_id'].str.contains(exclude_substr, na=False)]
            after_count = len(df_beta)
            print(f"Excluded participants containing '{exclude_substr}': {before_count - after_count} rows removed")
    
    # Add calorie column if specified
    if calorie is not None:
        df_beta['calorie'] = calorie
        print(f"\nAdded 'calorie' column with value: '{calorie}'")
    
    # Load LMER data
    print(f"\nLoading LMER data from: {lmer_file_path}")
    df_lmer = pd.read_csv(lmer_file_path, sep='\t')
    print(f"Loaded {len(df_lmer)} rows")
    
    # Function to convert participant_id format
    def from_afni_id(x):
        m = re.match(r"sub-([A-Z]+)(\d+)", x)
        if m:
            prefix = m.group(1).lower()
            number = m.group(2).zfill(3)
            return f"{prefix}{number}"
        return x
    
    # Convert participant_id to match LMER format
    df_beta['Subj'] = df_beta['participant_id'].apply(from_afni_id)
    
    # Rename columns to match LMER format
    df_beta_renamed = df_beta.rename(columns={
        'session': 'sess',
        'run': 'run'
    })
    
    print(f"\nUnique subjects in beta data: {df_beta_renamed['Subj'].nunique()}")
    print(f"Unique subjects in LMER data: {df_lmer['Subj'].nunique()}")
    
    # Merge on Subj, sess, and run to keep same structure as LMER
    df_merged = pd.merge(
        df_beta_renamed,
        df_lmer[['Subj', 'sess', 'run', 'sex', 'type_chx', 'handedness', 
                 'vas_debut', 'age_baseline', 'bmi_baseline']],
        on=['Subj', 'sess', 'run'],
        how='inner'
    )
    
    print(f"\nMerged dataframe rows: {len(df_merged)}")
    print(f"Unique subjects in merged df: {df_merged['Subj'].nunique()}")
    
    # Print mean beta summary - auto-detect ID column type
    print("\n" + "="*80)
    
    # Determine which ID column exists
    id_column = None
    if 'cluster_id' in df_merged.columns:
        id_column = 'cluster_id'
        print("Mean Beta by Cluster and Session:")
    elif 'network_id' in df_merged.columns:
        id_column = 'network_id'
        print("Mean Beta by Network and Session:")
    
    print("="*80)
    
    if id_column:
        mean_beta_summary = df_merged.groupby([id_column, 'sess']).agg({
            'mean_beta': ['mean', 'std', 'count']
        }).round(6)
        print(mean_beta_summary)
    else:
        print("Warning: Neither 'cluster_id' nor 'network_id' column found in merged dataframe")
    
    return df_merged, df_beta_renamed

## LMM function

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def run_lmm_analysis(df_merged, formula, show_diagnostics=True, show_boxplot=False, ylim=None, cluster_id=None, show_xlabel=True, show_legend=True, show_ylabel=True):
    """
    Run Linear Mixed Model analysis for each cluster or network.
    Parameters:
    -----------
    df_merged : pd.DataFrame
        Merged dataframe with beta and LMER data
    formula : str
        Formula for the mixed model (e.g., 'outcome ~ mean_beta + sex + age_baseline')
    show_diagnostics : bool, optional
        Whether to show diagnostic plots (default: True)
    show_boxplot : bool, optional
        Whether to show boxplot for interactions (default: False)
    ylim : tuple, optional
        Y-axis limits for boxplot as (ymin, ymax) (default: None for auto)
    cluster_id : int or list of int, optional
        Specific cluster/network ID(s) to analyze. If None, analyzes all (default: None)
    show_xlabel : bool, optional
        Whether to show x-axis labels on boxplot (default: True)
    show_ylabel : bool, optional
        Whether to show y-axis label on boxplot (default: True)
    show_legend : bool, optional
        Whether to show legend on boxplot (default: True)
    
    Returns:
    --------
    results_dict : dict
        Dictionary with cluster/network IDs as keys and model fit objects as values
    """
    
    # Extract outcome variable name from formula
    outcome_var = formula.split('~')[0].strip()
    
    results_dict = {}
    
    # Determine which ID column exists (cluster_id or network_id)
    id_column = None
    if 'cluster_id' in df_merged.columns:
        id_column = 'cluster_id'
        id_label = 'CLUSTER'
    elif 'network_id' in df_merged.columns:
        id_column = 'network_id'
        id_label = 'NETWORK'
    else:
        raise ValueError("Neither 'cluster_id' nor 'network_id' column found in dataframe")
    
    # Determine which IDs to analyze
    if cluster_id is None:
        id_values = sorted(df_merged[id_column].unique())
    elif isinstance(cluster_id, (list, tuple)):
        id_values = cluster_id
    else:
        id_values = [cluster_id]
    
    # Loop through specified IDs
    for cid in id_values:
        print(f"\n{'='*80}")
        print(f"{id_label} {cid} ANALYSIS")
        print(f"{'='*80}\n")
        
        # Filter data for current cluster/network
        df_cluster = df_merged[df_merged[id_column] == cid].copy()
        
        if len(df_cluster) == 0:
            print(f"Warning: {id_label} ID {cid} not found in dataframe. Skipping.\n")
            continue
        
        # Convert categorical variables to category dtype if they exist in formula
        categorical_vars = ['type_chx', 'sex', 'handedness', 'sess', 'calorie']
        for var in categorical_vars:
            if var in df_cluster.columns and f'C({var})' not in formula and var in formula:
                df_cluster[var] = df_cluster[var].astype('category')
        
        print(f"Model formula: {formula}")
        print(f"Data shape: {df_cluster.shape}")
        print(f"Unique subjects: {df_cluster['Subj'].nunique()}\n")
        
        try:
            # Fit the mixed model with random intercept per subject
            mixed_model = smf.mixedlm(formula, df_cluster, groups=df_cluster['Subj'], re_formula='1')
            mixed_model_fit = mixed_model.fit()
            
            # Store results
            results_dict[cid] = mixed_model_fit
            
            # Print the summary of the model
            print(mixed_model_fit.summary())
            
            # Print variance components
            random_effects_var = mixed_model_fit.cov_re.iloc[0, 0]
            residual_var = mixed_model_fit.scale
            icc = random_effects_var / (random_effects_var + residual_var)
            
            print(f"\n{'='*80}")
            print("VARIANCE COMPONENTS")
            print(f"{'='*80}")
            print(f"Random effects variance (τ²): {random_effects_var:.6f}")
            print(f"Residual variance (σ²): {residual_var:.6f}")
            print(f"ICC (intraclass correlation): {icc:.4f}")
            print(f"{'='*80}\n")
            
            # Boxplot for session effects (simpler version)
            if show_boxplot and 'sess' in formula and not ('calorie' in formula and '*' in formula):
                # Simple boxplot showing effect by session only
                session_labels = {
                    '1': 'Pre-surgery',
                    '2': '4 months\npost-surgery',
                    '3': '12 months\npost-surgery',
                    '4': '24 months\npost-surgery'
                }
                
                # Network color mapping if using network_id
                network_id_to_name = {
                    1: 'Vis',
                    2: 'SomMot',
                    3: 'DorsAttn',
                    4: 'SalVentAttn',
                    5: 'Limbic',
                    6: 'Cont',
                    7: 'Default'
                }
                
                network_colors = {
                    'Vis': '#4B0082',
                    'SomMot': '#228B22',
                    'DorsAttn': '#87CEEB',
                    'SalVentAttn': '#9370DB',
                    'Limbic': '#F5DEB3',
                    'Cont': '#FFA500',
                    'Default': '#DC143C'
                }
                
                # Determine boxplot color based on ID type
                if id_column == 'network_id' and cid in network_id_to_name:
                    network_name = network_id_to_name[cid]
                    box_color = network_colors.get(network_name, '#4472C4')
                else:
                    box_color = '#4472C4'  # default blue for clusters
                
                fig, ax = plt.subplots(figsize=(7, 4))
                
                # Prepare data for boxplot
                df_plot = df_cluster.copy()
                df_plot['sess'] = df_plot['sess'].astype(str)
                
                sessions = sorted(df_plot['sess'].unique())
                
                # Prepare boxplot data for each session
                boxplot_data = []
                positions = []
                
                for i, sess in enumerate(sessions, start=1):
                    data = df_plot[df_plot['sess'] == sess][outcome_var].values
                    if len(data) > 0:
                        boxplot_data.append(data)
                        positions.append(i)
                
                # Create boxplot with appropriate color
                bp = ax.boxplot(boxplot_data, positions=positions, widths=0.6,
                                patch_artist=True, 
                                boxprops=dict(facecolor=box_color, alpha=0.7),
                                medianprops=dict(color='black', linewidth=2),
                                whiskerprops=dict(color=box_color, linewidth=2),
                                capprops=dict(color=box_color, linewidth=2))
                
                # Customize plot
                ax.set_xticks(positions)
                
                if show_xlabel:
                    ax.set_xticklabels([session_labels.get(s, f'Session {s}') for s in sessions], 
                                    fontname='Aptos', fontsize=16)
                else:
                    ax.set_xticklabels([])
                    ax.tick_params(axis='x', bottom=False)
                
                if show_ylabel:
                    ax.set_ylabel('Effect (BOLD % change per $)', fontname='Aptos', fontsize=16)
                    ax.tick_params(axis='y', labelsize=16)
                    for label in ax.get_yticklabels():
                        label.set_fontname('Aptos')
                else:
                    ax.set_ylabel('')
                    ax.set_yticklabels([])
                    ax.tick_params(axis='y', left=False)
                
                # Set y-axis limits if specified
                if ylim is not None:
                    ax.set_ylim(ylim)
                
                # Remove top and right spines
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)
                
                # Set font for tick labels
                for label in ax.get_yticklabels():
                    label.set_fontname('Aptos')

                # Remove spines
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)

                if not show_ylabel:
                    ax.spines['left'].set_visible(False)

                if not show_xlabel:
                    ax.spines['bottom'].set_visible(False)
                
                plt.tight_layout()
                plt.show()

            # Boxplot for sess*calorie interaction (existing version)
            elif show_boxplot and 'sess' in formula and 'calorie' in formula and '*' in formula:
            
                # Session label mapping
                session_labels = {
                    '1': 'Pre-surgery',
                    '2': '4 months\npost-surgery',
                    '3': '12 months\npost-surgery',
                    '4': '24 months\npost-surgery'
                }
                
                fig, ax = plt.subplots(figsize=(7, 4))
                
                # Prepare data for boxplot
                df_plot = df_cluster.copy()
                df_plot['sess'] = df_plot['sess'].astype(str)
                
                # Define colors - using a brighter orange
                colors = {'high': '#FF8C00', 'low': 'blue'}  # DarkOrange instead of 'orange'
                
                # Create boxplot
                positions_high = [1, 5, 9, 13]  # Positions for high calorie
                positions_low = [2, 6, 10, 14]  # Positions for low calorie
                
                sessions = sorted(df_plot['sess'].unique())
                
                # Plot high calorie
                data_high = [df_plot[(df_plot['sess'] == sess) & (df_plot['calorie'] == 'high')][outcome_var].values 
                            for sess in sessions]
                bp_high = ax.boxplot(data_high, positions=positions_high, widths=0.6,
                                    patch_artist=True, 
                                    boxprops=dict(facecolor=colors['high'], alpha=0.7),
                                    medianprops=dict(color='black', linewidth=2),
                                    whiskerprops=dict(color=colors['high'], linewidth=2),
                                    capprops=dict(color=colors['high'], linewidth=2))
                
                # Plot low calorie
                data_low = [df_plot[(df_plot['sess'] == sess) & (df_plot['calorie'] == 'low')][outcome_var].values 
                        for sess in sessions]
                bp_low = ax.boxplot(data_low, positions=positions_low, widths=0.6,
                                    patch_artist=True,
                                    boxprops=dict(facecolor=colors['low'], alpha=0.7),
                                    medianprops=dict(color='black', linewidth=2),
                                    whiskerprops=dict(color=colors['low'], linewidth=2),
                                    capprops=dict(color=colors['low'], linewidth=2))
                
                # Customize plot
                ax.set_xticks([1.5, 5.5, 9.5, 13.5])
                
                # Show or hide x-axis labels
                if show_xlabel:
                    ax.set_xticklabels([session_labels[s] for s in sessions], fontname='Aptos', fontsize=16)
                else:
                    ax.set_xticklabels([])
                    ax.tick_params(axis='x', bottom=False)
                
                if show_ylabel:
                    ax.set_ylabel('Effect (BOLD % change per $)', fontname='Aptos', fontsize=16)
                    ax.tick_params(axis='y', labelsize=16)
                    for label in ax.get_yticklabels():
                        label.set_fontname('Aptos')
                else:
                    ax.set_ylabel('')
                    ax.set_yticklabels([])
                    ax.tick_params(axis='y', left=False)
                
                # Set y-axis limits if specified
                if ylim is not None:
                    ax.set_ylim(ylim)
                
                # Remove top and right spines
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)
                
                # Set font for tick labels
                for label in ax.get_yticklabels():
                    label.set_fontname('Aptos')
                
                # Add legend if requested
                if show_legend:
                    from matplotlib.patches import Patch
                    legend_elements = [Patch(facecolor=colors['high'], alpha=0.7, label='High Calorie'),
                                    Patch(facecolor=colors['low'], alpha=0.7, label='Low Calorie')]
                    legend = ax.legend(handles=legend_elements, loc='upper right', fontsize=16, prop={'family': 'Aptos'})

                # Remove spines
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)

                if not show_ylabel:
                    ax.spines['left'].set_visible(False)

                if not show_xlabel:
                    ax.spines['bottom'].set_visible(False)
                
                plt.tight_layout()
                plt.show()
            
            # Diagnostic plots
            if show_diagnostics:
                fig, axes = plt.subplots(2, 2, figsize=(14, 10))
                fig.suptitle(f'{id_label} {cid} - Diagnostic Plots', fontsize=16, fontweight='bold')
                
                # Residuals vs fitted values
                residuals = mixed_model_fit.resid
                fitted_values = mixed_model_fit.fittedvalues
                axes[0, 0].scatter(fitted_values, residuals, alpha=0.6)
                axes[0, 0].axhline(y=0, color='r', linestyle='--')
                axes[0, 0].set_xlabel('Fitted values', fontsize=11)
                axes[0, 0].set_ylabel('Residuals', fontsize=11)
                axes[0, 0].set_title('Residuals vs Fitted values', fontsize=12)
                axes[0, 0].grid(True, alpha=0.3)
                
                # QQ-plot of residuals
                sm.qqplot(residuals, line='45', ax=axes[0, 1])
                axes[0, 1].set_title('QQ-plot of Residuals', fontsize=12)
                axes[0, 1].grid(True, alpha=0.3)
                
                # Histogram of residuals
                axes[1, 0].hist(residuals, bins=20, edgecolor='black', alpha=0.7)
                axes[1, 0].set_xlabel('Residuals', fontsize=11)
                axes[1, 0].set_ylabel('Frequency', fontsize=11)
                axes[1, 0].set_title('Histogram of Residuals', fontsize=12)
                axes[1, 0].grid(True, alpha=0.3)
                
                # Scale-Location plot
                standardized_residuals = np.sqrt(np.abs(residuals / np.std(residuals)))
                axes[1, 1].scatter(fitted_values, standardized_residuals, alpha=0.6)
                axes[1, 1].set_xlabel('Fitted values', fontsize=11)
                axes[1, 1].set_ylabel('√|Standardized Residuals|', fontsize=11)
                axes[1, 1].set_title('Scale-Location Plot', fontsize=12)
                axes[1, 1].grid(True, alpha=0.3)
                
                plt.tight_layout()
                plt.show()
            
            # Breusch-Pagan test for homoscedasticity
            bp_test = het_breuschpagan(mixed_model_fit.resid, mixed_model_fit.model.exog)
            bp_pvalue = bp_test[1]
            print(f"\nBreusch-Pagan test p-value: {bp_pvalue:.6f}")
            
            if bp_pvalue < 0.05:
                print("Heteroscedasticity detected. The variance of residuals is not constant.")
            else:
                print("No heteroscedasticity detected. The variance of residuals is constant.")
            
            # Shapiro-Wilk test for normality of residuals
            shapiro_test = stats.shapiro(mixed_model_fit.resid)
            print(f"\nShapiro-Wilk test p-value: {shapiro_test.pvalue:.6f}")
            
            if shapiro_test.pvalue < 0.05:
                print("Residuals are not normally distributed.")
            else:
                print("Residuals are normally distributed.")
            
            # Calculate VIF for each predictor
            print("\nVariance Inflation Factors:")
            vif_data = pd.DataFrame()
            vif_data["feature"] = mixed_model_fit.model.exog_names
            vif_data["VIF"] = [variance_inflation_factor(mixed_model_fit.model.exog, i) 
                               for i in range(mixed_model_fit.model.exog.shape[1])]
            print(vif_data)
            
            # Model fit statistics
            k = len(mixed_model_fit.params)
            n = mixed_model_fit.nobs
            log_likelihood = mixed_model_fit.llf
            aic = 2 * k - 2 * log_likelihood
            bic = np.log(n) * k - 2 * log_likelihood
            
            print(f"\nAIC: {aic:.4f}")
            print(f"BIC: {bic:.4f}")
            
            # Wald test results
            print("\nWald test results:")
            wald_test = mixed_model_fit.wald_test_terms()
            pd.set_option('display.float_format', lambda x: '%.10f' % x)
            print(wald_test)
            
            # Calculate R-squared for each parameter
            print("\nR-squared for each parameter:")
            r_squared_data = pd.DataFrame(columns=["Parameter", "R_squared"])
            
            for param in mixed_model_fit.model.exog_names:
                param_base = param.split('[')[0]
                
                if param_base == 'Intercept':
                    continue
                
                try:
                    # Handle categorical variable
                    if param_base.startswith('C('):
                        # Extract variable name from C(var)
                        var_name = param_base.split('C(')[1].split(')')[0]
                        formula_single = f'{outcome_var} ~ C({var_name})'
                    else:
                        formula_single = f'{outcome_var} ~ {param_base}'
                    
                    single_model = smf.mixedlm(formula_single, df_cluster, 
                                              groups=df_cluster['Subj'], re_formula='1')
                    single_model_fit = single_model.fit()
                    
                    # Calculate the R-squared as the proportion of variance explained
                    total_variance = np.var(df_cluster[outcome_var])
                    explained_variance = total_variance - np.var(single_model_fit.resid)
                    r_squared = explained_variance / total_variance
                    
                    # Append the result
                    r_squared_data = pd.concat([r_squared_data, 
                                               pd.DataFrame([{"Parameter": param_base, 
                                                             "R_squared": r_squared * 100}])], 
                                              ignore_index=True)
                except Exception as e:
                    print(f"Could not calculate R-squared for {param_base}: {str(e)}")
            
            print(r_squared_data)
            print("\n")
            
        except Exception as e:
            print(f"Error fitting model for {id_label} {cid}: {str(e)}\n")
    
    return results_dict

## nilearn slice figure function

In [ ]:
from nilearn.plotting import plot_stat_map, show
from nilearn import image
from nilearn.image import coord_transform
import matplotlib.pyplot as plt
import numpy as np

def plot_cluster_slice(
    df_merged,
    cluster_id,
    effect_map_path,
    cluster_map_path,
    title=None,
    cmap='coolwarm',
    vmin=-0.15,
    vmax=0.15,
    threshold=0.01,
    alpha=0.7,
    cut_coords=None,
    display_mode='ortho',
    figure_width=7,
    output_file=None
):
    """
    Plot brain slices for a specific cluster showing effect size map with cluster contours.
    
    Parameters:
    -----------
    df_merged : pd.DataFrame
        Dataframe containing cluster coordinates (must have com_mni_x, com_mni_y, com_mni_z columns)
    cluster_id : int
        Cluster ID to plot
    effect_map_path : str
        Path to effect size NIfTI file
    cluster_map_path : str
        Path to cluster mask NIfTI file
    title : str, optional
        Title for the plot (default: None, no title displayed)
    cmap : str, optional
        Colormap name (default: 'coolwarm')
    vmin : float, optional
        Minimum value for colormap (default: -0.15)
    vmax : float, optional
        Maximum value for colormap (default: 0.15)
    threshold : float, optional
        Threshold for transparency (default: 0.01)
    alpha : float, optional
        Transparency level (default: 0.7)
    cut_coords : tuple or float, optional
        For 'ortho': tuple (x, y, z). For 'x'/'y'/'z': single coordinate value.
        If None, uses cluster center of mass.
    display_mode : str, optional
        Display mode: 'ortho' for 3 views, 'x' for sagittal, 'y' for coronal, 'z' for axial (default: 'ortho')
    figure_width : float, optional
        Width of figure in inches (default: 6)
    output_file : str, optional
        Path to save figure. If None, displays but doesn't save.
    
    Returns:
    --------
    fig : matplotlib.figure.Figure
        The figure object
    """
    
    # Filter dataframe for the cluster
    cluster_df = df_merged[df_merged['cluster_id'] == cluster_id]
    
    if len(cluster_df) == 0:
        raise ValueError(f"Cluster ID {cluster_id} not found in dataframe")
    
    # Get cluster center of mass coordinates
    com_x = cluster_df['com_mni_x'].iloc[0]
    com_y = cluster_df['com_mni_y'].iloc[0]
    com_z = cluster_df['com_mni_z'].iloc[0]
    
    # Set cut_coords based on display_mode
    if cut_coords is None:
        if display_mode == 'ortho':
            cut_coords = (com_x, com_y, com_z)
        elif display_mode == 'x':
            cut_coords = [com_x]  # Wrap in list for single slice
        elif display_mode == 'y':
            cut_coords = [com_y]  # Wrap in list for single slice
        elif display_mode == 'z':
            cut_coords = [com_z]  # Wrap in list for single slice
        else:
            cut_coords = (com_x, com_y, com_z)
    elif display_mode in ['x', 'y', 'z'] and not isinstance(cut_coords, list):
        # If user provides a single coordinate, ensure it's in a list
        cut_coords = [cut_coords]
    
    # Load images
    effect_img = image.load_img(effect_map_path)
    cluster_img = image.load_img(cluster_map_path)
    
    # Create mask for specific cluster
    cluster_data = cluster_img.get_fdata()
    cluster_mask = (cluster_data == cluster_id).astype(float)
    cluster_mask_img = image.new_img_like(cluster_img, cluster_mask)
    
    # Calculate figure height based on display mode
    if display_mode == 'ortho':
        fig_height = figure_width * 0.4
    else:
        fig_height = figure_width
    
    # Create figure
    fig, axes = plt.subplots(figsize=(figure_width, fig_height), dpi=300)
    
    # Plot stat map with transparency (no colorbar, no title)
    display = plot_stat_map(
        effect_img,
        title=title,
        cut_coords=cut_coords,
        display_mode=display_mode,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        threshold=threshold,
        alpha=alpha,
        axes=axes,
        colorbar=False,
        annotate=True,
        draw_cross=False,
        black_bg=False
    )
    
    # Increase font size for annotations
    for cut_ax in display.axes.values():
        for text in cut_ax.ax.texts:
            text.set_fontsize(16)
    
    # Add cluster contours in black
    display.add_contours(
        cluster_mask_img,
        filled=False,
        colors=['black'],
        linewidths=2,
        levels=[0.5]
    )
    
    # Calculate and print cluster statistics
    effect_data = effect_img.get_fdata()
    effect_vals = effect_data[cluster_data == cluster_id]
    mean_effect = np.mean(effect_vals)
    
    print(f"Cluster {cluster_id}:")
    print(f"  Center of Mass (x, y, z): ({com_x}, {com_y}, {com_z})")
    if display_mode == 'ortho':
        print(f"  Cut coordinates: {cut_coords}")
    else:
        print(f"  Cut coordinate ({display_mode}): {cut_coords[0] if isinstance(cut_coords, list) else cut_coords}")
    print(f"  Mean Effect Size: {mean_effect:.4f}")
    print(f"  Number of voxels: {len(effect_vals)}")
    
    plt.tight_layout()
    
    if output_file:
        plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"Figure saved to: {output_file}")
    
    show()
    
    return fig

# Example usage:
# Ortho view (3 slices):
# fig = plot_cluster_slice(
#     df_merged=df_merged_Lo,
#     cluster_id=1,
#     effect_map_path='/path/to/effect_map.nii.gz',
#     cluster_map_path='/path/to/cluster_map.nii.gz',
#     display_mode='ortho',
#     vmin=-0.15,
#     vmax=0.15
# )

# Single axial slice (z):
# fig = plot_cluster_slice(
#     df_merged=df_merged_Lo,
#     cluster_id=1,
#     effect_map_path='/path/to/effect_map.nii.gz',
#     cluster_map_path='/path/to/cluster_map.nii.gz',
#     display_mode='z',
#     vmin=-0.15,
#     vmax=0.15
# )

# Single sagittal slice (x) with custom coordinate:
# fig = plot_cluster_slice(
#     df_merged=df_merged_Lo,
#     cluster_id=1,
#     effect_map_path='/path/to/effect_map.nii.gz',
#     cluster_map_path='/path/to/cluster_map.nii.gz',
#     display_mode='x',
#     cut_coords=10,  # Can provide as single number, will be wrapped in list
#     vmin=-0.15,
#     vmax=0.15
# )

## yeo parcelattion (debug)

In [ ]:
from nilearn import datasets, image
from nilearn.image import resample_to_img
from nilearn.plotting import plot_roi
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def plot_yeo_network_effects_multisession_separate_debug(
    effect_map_paths,
    n_networks=7,
    session_labels=None,
    output_dir=None,
    ylim=None,
    figsize=(8, 5),
    show_atlas_debug=True,
    show_xlabel=True,
    show_ylabel=True,
    ytick_count=5
):
    """
    Plot effect sizes for each Yeo network across multiple sessions.
    Creates one figure per network with FIXED plot area dimensions.
    
    Parameters:
    -----------
    effect_map_paths : dict
        Dictionary mapping session numbers to effect map paths
    n_networks : int, optional
        Number of Yeo networks (7 or 17) (default: 7)
    session_labels : dict, optional
        Dictionary mapping session numbers to labels
    output_dir : str, optional
        Directory to save figures. If None, doesn't save.
    ylim : tuple, optional
        Y-axis limits as (ymin, ymax)
    figsize : tuple, optional
        Figure size (width, height)
    show_atlas_debug : bool, optional
        Whether to show atlas debug plots (default: True)
    show_xlabel : bool, optional
        Whether to show x-axis labels and tick labels (default: True)
    show_ylabel : bool, optional
        Whether to show y-axis label and tick labels (default: True)
    
    Returns:
    --------
    df_network_effects : pd.DataFrame
        DataFrame with network effects for all sessions
    figures : dict
        Dictionary with network names as keys and figure objects as values
    """
    
    # Define network colors
    network_colors = {
        'Vis': '#4B0082',
        'SomMot': '#228B22',
        'DorsAttn': '#87CEEB',
        'SalVentAttn': '#9370DB',
        'Limbic': '#F5DEB3',
        'Cont': '#FFA500',
        'Default': '#DC143C'
    }
    
    # Fetch Schaefer atlas
    print(f"Fetching Schaefer atlas with {n_networks} Yeo networks...")
    atlas = datasets.fetch_atlas_schaefer_2018(
        n_rois=400,
        yeo_networks=n_networks,
        resolution_mm=1
    )
    
    atlas_img = image.load_img(atlas['maps'])
    
    print(f"\n{'='*80}")
    print("ATLAS INFO (Target resolution):")
    print(f"{'='*80}")
    print(f"Shape: {atlas_img.shape}")
    print(f"Affine:\n{atlas_img.affine}")
    print(f"Voxel sizes: {atlas_img.header.get_zooms()}")
    
    import nibabel as nib
    atlas_ornt = nib.orientations.aff2axcodes(atlas_img.affine)
    print(f"Atlas orientation: {atlas_ornt}")
    
    # Parse unique networks from ROI labels
    unique_networks = []
    network_roi_mapping = {}
    
    for i, label in enumerate(atlas['labels']):
        label_str = label.decode('utf-8') if isinstance(label, bytes) else label
        network = label_str.split('_')[2]
        
        if network not in unique_networks:
            unique_networks.append(network)
            network_roi_mapping[network] = []
        
        network_roi_mapping[network].append(i + 1)
    
    print(f"\nFound {len(unique_networks)} unique networks:")
    for net in unique_networks:
        print(f"  {net}: {len(network_roi_mapping[net])} ROIs")
    
    # Collect all effect values
    all_data = []
    
    for sess_idx, (sess_num, effect_path) in enumerate(sorted(effect_map_paths.items())):
        print(f"\n{'='*80}")
        print(f"Processing Session {sess_num}...")
        print(f"{'='*80}")
        print(f"  Loading effect map from: {effect_path}")
        
        effect_img = image.load_img(effect_path)
        
        print(f"\n  ORIGINAL EFFECT MAP INFO:")
        print(f"  Shape: {effect_img.shape}")
        print(f"  Affine:\n{effect_img.affine}")
        print(f"  Voxel sizes: {effect_img.header.get_zooms()}")
        
        effect_ornt = nib.orientations.aff2axcodes(effect_img.affine)
        print(f"  Effect map orientation: {effect_ornt}")
        
        # Resample effect map
        print(f"\n  Resampling effect map to 1mm to match atlas...")
        effect_resampled = resample_to_img(
            effect_img,
            atlas_img,
            interpolation='continuous'
        )
        
        print(f"\n  RESAMPLED EFFECT MAP INFO:")
        print(f"  Shape: {effect_resampled.shape}")
        print(f"  Affine:\n{effect_resampled.affine}")
        print(f"  Voxel sizes: {effect_resampled.header.get_zooms()}")
        
        resamp_ornt = nib.orientations.aff2axcodes(effect_resampled.affine)
        print(f"  Resampled effect map orientation: {resamp_ornt}")
        
        if effect_ornt != resamp_ornt:
            print(f"\n  WARNING: Orientation change!")
            print(f"    Original: {effect_ornt}")
            print(f"    Resampled: {resamp_ornt}")
        
        # Show debug plots only for first session
        if show_atlas_debug and sess_idx == 0:
            print("\n  Generating resampling debug plots...")
            
            mni_x, mni_y, mni_z = 0, 0, 0
            
            from nilearn.image import coord_transform
            vox_x_orig, vox_y_orig, vox_z_orig = coord_transform(
                mni_x, mni_y, mni_z, np.linalg.inv(effect_img.affine)
            )
            vox_x_orig, vox_y_orig, vox_z_orig = int(round(vox_x_orig)), int(round(vox_y_orig)), int(round(vox_z_orig))
            
            vox_x_resamp, vox_y_resamp, vox_z_resamp = coord_transform(
                mni_x, mni_y, mni_z, np.linalg.inv(effect_resampled.affine)
            )
            vox_x_resamp, vox_y_resamp, vox_z_resamp = int(round(vox_x_resamp)), int(round(vox_y_resamp)), int(round(vox_z_resamp))
            
            print(f"\n  MNI coordinates: ({mni_x}, {mni_y}, {mni_z})")
            print(f"  Original effect map voxel: ({vox_x_orig}, {vox_y_orig}, {vox_z_orig})")
            print(f"  Resampled effect map voxel: ({vox_x_resamp}, {vox_y_resamp}, {vox_z_resamp})")
            
            effect_data_orig = effect_img.get_fdata()
            effect_data_resamp = effect_resampled.get_fdata()
            
            print(f"\n  VALUES AT MNI (0, 0, 0):")
            print(f"  Original effect map value: {effect_data_orig[vox_x_orig, vox_y_orig, vox_z_orig]:.6f}")
            print(f"  Resampled effect map value: {effect_data_resamp[vox_x_resamp, vox_y_resamp, vox_z_resamp]:.6f}")
            
            fig, axes = plt.subplots(2, 3, figsize=(18, 12))
            fig.suptitle(f'Effect Map: Before (3mm) and After (1mm) Resampling (MNI: {mni_x}, {mni_y}, {mni_z})', 
                        fontsize=16, fontweight='bold')
            
            # Original effect map - three views
            axes[0, 0].imshow(effect_data_orig[vox_x_orig, :, :].T, cmap='coolwarm', origin='lower', vmin=-0.1, vmax=0.1)
            axes[0, 0].set_title(f'Original 3mm - Sagittal (MNI x={mni_x}, vox x={vox_x_orig})', fontsize=12)
            axes[0, 0].axis('off')
            
            axes[0, 1].imshow(effect_data_orig[:, vox_y_orig, :].T, cmap='coolwarm', origin='lower', vmin=-0.1, vmax=0.1)
            axes[0, 1].set_title(f'Original 3mm - Coronal (MNI y={mni_y}, vox y={vox_y_orig})', fontsize=12)
            axes[0, 1].axis('off')
            
            axes[0, 2].imshow(effect_data_orig[:, :, vox_z_orig].T, cmap='coolwarm', origin='lower', vmin=-0.1, vmax=0.1)
            axes[0, 2].set_title(f'Original 3mm - Axial (MNI z={mni_z}, vox z={vox_z_orig})', fontsize=12)
            axes[0, 2].axis('off')
            
            # Resampled effect map - three views
            axes[1, 0].imshow(effect_data_resamp[vox_x_resamp, :, :].T, cmap='coolwarm', origin='lower', vmin=-0.1, vmax=0.1)
            axes[1, 0].set_title(f'Resampled 1mm - Sagittal (MNI x={mni_x}, vox x={vox_x_resamp})', fontsize=12)
            axes[1, 0].axis('off')
            
            axes[1, 1].imshow(effect_data_resamp[:, vox_y_resamp, :].T, cmap='coolwarm', origin='lower', vmin=-0.1, vmax=0.1)
            axes[1, 1].set_title(f'Resampled 1mm - Coronal (MNI y={mni_y}, vox y={vox_y_resamp})', fontsize=12)
            axes[1, 1].axis('off')
            
            axes[1, 2].imshow(effect_data_resamp[:, :, vox_z_resamp].T, cmap='coolwarm', origin='lower', vmin=-0.1, vmax=0.1)
            axes[1, 2].set_title(f'Resampled 1mm - Axial (MNI z={mni_z}, vox z={vox_z_resamp})', fontsize=12)
            axes[1, 2].axis('off')
            
            plt.tight_layout()
            plt.show()
            
            print(f"\n  EFFECT MAP STATISTICS:")
            print(f"  Original (3mm) - Non-zero voxels: {np.count_nonzero(effect_data_orig)}")
            print(f"  Original (3mm) - Mean: {np.nanmean(effect_data_orig):.6f}")
            print(f"  Resampled (1mm) - Non-zero voxels: {np.count_nonzero(effect_data_resamp)}")
            print(f"  Resampled (1mm) - Mean: {np.nanmean(effect_data_resamp):.6f}")
        
        # Get data arrays
        effect_data = effect_resampled.get_fdata()
        atlas_data = atlas_img.get_fdata().astype(int)
        
        print(f"\n  FINAL EFFECT DATA STATISTICS (1mm):")
        print(f"  Min: {np.nanmin(effect_data):.6f}")
        print(f"  Max: {np.nanmax(effect_data):.6f}")
        print(f"  Mean: {np.nanmean(effect_data):.6f}")
        print(f"  Non-zero voxels: {np.count_nonzero(effect_data)}")
        print(f"  NaN voxels: {np.isnan(effect_data).sum()}")
        
        # Extract effect values for each network
        for network in unique_networks:
            roi_indices = network_roi_mapping[network]
            network_mask = np.isin(atlas_data, roi_indices)
            effect_values = effect_data[network_mask]
            
            print(f"\n  {network}:")
            print(f"    Mask voxels: {network_mask.sum()}")
            print(f"    Effect values before filtering: {len(effect_values)}")
            
            effect_values = effect_values[~np.isnan(effect_values)]
            print(f"    After removing NaN: {len(effect_values)}")
            effect_values = effect_values[effect_values != 0]
            print(f"    After removing zeros: {len(effect_values)}")
            
            for val in effect_values:
                all_data.append({
                    'session': sess_num,
                    'network': network,
                    'effect_size': val
                })
            
            if len(effect_values) > 0:
                print(f"    Mean effect: {np.mean(effect_values):.6f}")
                print(f"    Std effect: {np.std(effect_values):.6f}")
    
    # Create DataFrame
    df_network_effects = pd.DataFrame(all_data)
    
    print("\n" + "="*80)
    print("Summary Statistics by Network and Session:")
    print("="*80)
    summary = df_network_effects.groupby(['network', 'session'])['effect_size'].agg(['mean', 'std', 'count'])
    print(summary)
    
    # Default session labels
    if session_labels is None:
        session_labels = {
            1: 'Pre-surgery',
            2: '4 months\npost-surgery',
            3: '12 months\npost-surgery',
            4: '24 months\npost-surgery'
        }
    
    sessions = sorted(df_network_effects['session'].unique())
    
    # Network name mapping
    network_full_names = {
        'Vis': 'Visual Network',
        'SomMot': 'Somatomotor Network',
        'DorsAttn': 'Dorsal Attention Network',
        'SalVentAttn': 'Ventral Attention Network',
        'Limbic': 'Limbic Network',
        'Cont': 'Frontoparietal Control Network',
        'Default': 'Default Mode Network'
    }
    
    # FIXED SUBPLOT POSITIONING - ensures identical plot areas
    # Define absolute positions [left, bottom, width, height] in figure coordinates (0-1)
    # These remain constant regardless of axis visibility
    plot_left = 0.15 if show_ylabel else 0.05
    plot_bottom = 0.15 if show_xlabel else 0.10
    plot_width = 0.80
    plot_height = 0.75
    
    # Create one figure per network
    figures = {}
    
    for network in unique_networks:
        print(f"\nCreating figure for {network}...")
        
        # Create figure with fixed size
        fig = plt.figure(figsize=figsize, dpi=300)
        
        # Add subplot with FIXED absolute positioning
        ax = fig.add_axes([plot_left, plot_bottom, plot_width, plot_height])
        
        # Prepare data for this network
        network_data = df_network_effects[df_network_effects['network'] == network]
        
        # Create boxplot data
        boxplot_data = []
        positions = []
        
        for i, sess in enumerate(sessions, start=1):
            data = network_data[network_data['session'] == sess]['effect_size'].values
            if len(data) > 0:
                boxplot_data.append(data)
                positions.append(i)
        
        # Get color for this network
        color = network_colors.get(network, '#808080')
        
        # Create boxplot
        bp = ax.boxplot(
            boxplot_data,
            positions=positions,
            widths=0.6,
            patch_artist=True,
            showfliers=False,
            medianprops=dict(color='black', linewidth=2.5),
            boxprops=dict(linewidth=1.5),
            whiskerprops=dict(linewidth=1.5),
            capprops=dict(linewidth=1.5)
        )
        
        # Color the boxes
        for patch in bp['boxes']:
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        # Set x-axis limits to maintain consistent spacing
        ax.set_xlim(0.5, len(sessions) + 0.5)
        
        # Customize x-axis
        ax.set_xticks(positions)
        
        if show_xlabel:
            ax.set_xticklabels([session_labels.get(s, f'Session {s}') for s in sessions], 
                               fontname='Aptos', fontsize=14)
        else:
            ax.set_xticklabels([])
            ax.tick_params(axis='x', which='both', length=0)
        
        # ALWAYS set y-axis limits
        if ylim is not None:
            ax.set_ylim(ylim)
            ymin, ymax = ylim
            if ytick_count is not None and ytick_count >= 2:
                yticks = np.linspace(ymin, ymax, ytick_count)
                ax.set_yticks(yticks)
        
        # Customize y-axis
        if show_ylabel:
            ax.set_ylabel('Effect Size (BOLD % change)', fontname='Aptos', fontsize=16)
            ax.tick_params(axis='y', labelsize=14)
            ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.2f}'))
            for label in ax.get_yticklabels():
                label.set_fontname('Aptos')
        else:
            ax.set_ylabel('')
            ax.set_yticklabels([])
            ax.tick_params(axis='y', which='both', length=0)
        
        # Set title
        full_name = network_full_names.get(network, network)
        ax.set_title(f'{full_name}', fontname='Aptos', fontsize=18, fontweight='bold', pad=20)
        
        # Add horizontal line at y=0
        ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
        
        # Remove spines
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
        if not show_ylabel:
            ax.spines['left'].set_visible(False)
        
        if not show_xlabel:
            ax.spines['bottom'].set_visible(False)
        
        # NO plt.tight_layout() - this would change subplot positions!
        
        # Save figure
        if output_dir:
            import os
            os.makedirs(output_dir, exist_ok=True)
            output_file = os.path.join(output_dir, f'yeo_network_{network}_across_sessions.png')
            plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white', pad_inches=0.1)
            print(f"Figure saved to: {output_file}")
        
        plt.show()
        
        figures[network] = fig
    
    return df_network_effects, figures

# Results

In [ ]:
#Example usage:
df_merged_Lo, df_beta_Lo = load_and_merge_beta_lmer_data(
    beta_file_path='/home/gagpat01/mes_projets/maitrise/fmri/results/beta_estimates_sess2vs1_clustered_p001_ACF_residuals_mod_hi_vs_mod_lo_v7_literature_complex_psc_qc_final_20260226_172623.tsv',
    lmer_file_path='/home/gagpat01/mes_projets/maitrise/fmri/results/3dlmerdata_table_with_covariates_mod_hi_vs_mod_lo_v7_literature_complex_psc_qc_final.txt',
    exclude_participants=['RGC'],
    calorie='low'
)

In [ ]:
#Example usage:
df_merged_Hi, df_beta_Hi = load_and_merge_beta_lmer_data(
    beta_file_path='/home/gagpat01/mes_projets/maitrise/fmri/results/beta_estimates_sess2vs1_clustered_p001_ACF_residuals_mod_hi_vs_mod_lo_v7_literature_complex_psc_qc_final_20260226_171255.tsv',
    lmer_file_path='/home/gagpat01/mes_projets/maitrise/fmri/results/3dlmerdata_table_with_covariates_mod_hi_vs_mod_lo_v7_literature_complex_psc_qc_final.txt',
    exclude_participants=['RGC'],
    calorie='high'
)

In [ ]:
#Example usage:
df_merged_Hi_v_Lo, df_beta_Hi_v_Lo = load_and_merge_beta_lmer_data(
    beta_file_path='/home/gagpat01/mes_projets/maitrise/fmri/results/beta_estimates_sess2vs1_clustered_p001_ACF_residuals_mod_hi_vs_mod_lo_v7_literature_complex_psc_qc_final_20260226_181407.tsv',
    lmer_file_path='/home/gagpat01/mes_projets/maitrise/fmri/results/3dlmerdata_table_with_covariates_mod_hi_vs_mod_lo_v7_literature_complex_psc_qc_final.txt',
    exclude_participants=['RGC']
)

In [ ]:
results = run_lmm_analysis(
    df_merged_Hi, 
    formula='mean_beta ~  sess + sex + age_baseline + bmi_baseline + C(type_chx) + handedness',
    show_diagnostics=False

)

In [ ]:
results = run_lmm_analysis(
    df_merged_Lo, 
    formula='mean_beta ~  sess + sex + age_baseline + bmi_baseline + C(type_chx) + handedness',
    show_diagnostics=True
)

In [ ]:
results = run_lmm_analysis(
    df_merged_Hi_v_Lo, 
    formula='mean_beta ~  sess + sex + age_baseline + bmi_baseline + C(type_chx) + handedness',
    show_diagnostics=True
)

In [ ]:
df_merged_concat_Hi_Lo = pd.concat([df_merged_Hi, df_merged_Lo], ignore_index=True)

In [ ]:
results = run_lmm_analysis(
    df_merged_concat_Hi_Lo, 
    formula='mean_beta ~  sess*calorie + sex + age_baseline + bmi_baseline + C(type_chx) + handedness',
    show_diagnostics=True
)

## Cluster 1 dlpfc

In [ ]:
results = run_lmm_analysis(
    df_merged_concat_Hi_Lo, 
    formula='mean_beta ~ sess*calorie + sex + age_baseline + bmi_baseline + C(type_chx) + handedness',
    show_diagnostics=True,
    show_boxplot=True,
    ylim=(-0.7, 0.7),  # Specify y-axis range
    show_xlabel = True
)

In [ ]:
fig = plot_cluster_slice(
    df_merged=df_merged_Lo,
    cluster_id=1,
    effect_map_path='/home/gagpat01/mes_projets/maitrise/fmri/results/3dlmer/effect_maps_mod_hi_vs_mod_lo_v7_literature_complex_psc_qc_final/mod_hi_vs_mod_lo_v7_literature_complex_psc_qc_final_sess2_vs_sess1.nii.gz',
    cluster_map_path='/home/gagpat01/mes_projets/maitrise/fmri/results/3dlmer/results/mod_hi_vs_mod_lo_v7_literature_complex_psc_qc_final/sess2vs1_clustered_p001_ACF_residuals_mod_hi_vs_mod_lo_v7_literature_complex_psc_qc_final.nii.gz',
    vmin=-0.15,
    vmax=0.15,
    display_mode='z',
    figure_width=3,
)

## Cluster 2 parietal cortex

In [ ]:
results = run_lmm_analysis(
    df_merged_concat_Hi_Lo, 
    formula='mean_beta ~ sess*calorie + sex + age_baseline + bmi_baseline + C(type_chx) + handedness',
    show_diagnostics=True,
    show_boxplot=True,
    ylim=(-0.7, 0.7), # Specify y-axis range
    cluster_id = 2, 
    show_legend = False
)

In [ ]:
fig = plot_cluster_slice(
    df_merged=df_merged_Lo,
    cluster_id=2,
    effect_map_path='/home/gagpat01/mes_projets/maitrise/fmri/results/3dlmer/effect_maps_mod_hi_vs_mod_lo_v7_literature_complex_psc_qc_final/mod_hi_vs_mod_lo_v7_literature_complex_psc_qc_final_sess2_vs_sess1.nii.gz',
    cluster_map_path='/home/gagpat01/mes_projets/maitrise/fmri/results/3dlmer/results/mod_hi_vs_mod_lo_v7_literature_complex_psc_qc_final/sess2vs1_clustered_p001_ACF_residuals_mod_hi_vs_mod_lo_v7_literature_complex_psc_qc_final.nii.gz',
    vmin=-0.15,
    vmax=0.15,
    display_mode='z',
    figure_width=3,
)

## yeo with first level

In [ ]:
#Example usage:
df_merged_Hi_v_Lo_Yeo, df_beta_Hi_v_Lo_Yeo = load_and_merge_beta_lmer_data(
    beta_file_path='/home/gagpat01/mes_projets/maitrise/fmri/results/beta_estimates_yeo7-7networks_20260409_145148.tsv',
    lmer_file_path='/home/gagpat01/mes_projets/maitrise/fmri/results/3dlmerdata_table_with_covariates_mod_hi_vs_mod_lo_v7_literature_complex_psc_qc_final.txt',
    exclude_participants=['RGC']
)


In [ ]:
results = run_lmm_analysis(
    df_merged_Hi_v_Lo_Yeo, 
    formula='mean_beta ~ sess + sex + age_baseline + bmi_baseline + C(type_chx) + handedness',
    show_diagnostics=False,
    show_boxplot=True,
    ylim=(-0.8, 0.8),
    show_xlabel=True,
    show_ylabel=True

)